# Feature Engineering

In this notebook, meaningful features are created from the cleaned groundwater monitoring dataset to improve machine learning model performance.

The engineered features capture temporal patterns, historical groundwater behaviour, spatial information, and station-specific characteristics. These features will later be used for training and evaluating predictive models.

The feature engineering process includes:

- Date and time feature extraction
- Cyclical encoding of temporal variables
- Lag features
- Rolling statistical features
- Trend-based features
- Station encoding
- Spatial features
- Preparation of the final machine learning dataset

## 1. Import Required Libraries

The necessary Python libraries for data manipulation, visualization, feature engineering, and machine learning are imported.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

## 2. Load the Cleaned Dataset

The cleaned groundwater dataset generated during the preprocessing stage is loaded. The timestamp column is converted to datetime format, and the dataset is sorted chronologically for each monitoring station to ensure correct feature generation.

In [3]:
# Load cleaned dataset

df = pd.read_csv(
    "/content/groundwater_cleaned.csv",
    parse_dates=["Data Acquisition Time"]
)

# Sort chronologically within each station
df = df.sort_values(
    ["Station", "Data Acquisition Time"]
).reset_index(drop=True)

print("Dataset Shape:", df.shape)
print("Stations:", df["Station"].nunique())

df.head()

Dataset Shape: (99445, 7)
Stations: 25


,Station,Data Acquisition Time,Latitude,Longitude,Groundwater Level Telemetry 6 Hourly (meter),RL_MSL,time_diff
0,Adakamaranahalli,2021-09-27 00:00:00,13.07,77.45,-22.51,849.0,NaN
1,Adakamaranahalli,2021-09-27 06:00:00,13.07,77.45,-22.87,849.0,0 days 06:00:00
2,Adakamaranahalli,2021-09-27 12:00:00,13.07,77.45,-23.23,849.0,0 days 06:00:00
3,Adakamaranahalli,2021-09-27 18:00:00,13.07,77.45,-22.66,849.0,0 days 06:00:00
4,Adakamaranahalli,2021-09-28 00:00:00,13.07,77.45,-22.46,849.0,0 days 06:00:00


## 3. Date and Time Features

Temporal information is extracted from the timestamp column to capture daily, monthly, quarterly, and yearly groundwater patterns.

The following features are created:

- Year
- Month
- Day
- Day of Week
- Week of Year
- Quarter
- Is Weekend

In [4]:
# Create temporal features

df["Year"] = df["Data Acquisition Time"].dt.year

df["Month"] = df["Data Acquisition Time"].dt.month

df["Day"] = df["Data Acquisition Time"].dt.day

df["Hour"] = df["Data Acquisition Time"].dt.hour

df["DayOfWeek"] = df["Data Acquisition Time"].dt.dayofweek

df["WeekOfYear"] = (
    df["Data Acquisition Time"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df["Quarter"] = df["Data Acquisition Time"].dt.quarter

df["IsWeekend"] = (
    df["DayOfWeek"] >= 5
).astype(int)

print("Temporal features created successfully.")

df.head()

Temporal features created successfully.


,Station,Data Acquisition Time,Latitude,Longitude,Groundwater Level Telemetry 6 Hourly (meter),RL_MSL,time_diff,Year,Month,Day,Hour,DayOfWeek,WeekOfYear,Quarter,IsWeekend
0,Adakamaranahalli,2021-09-27 00:00:00,13.07,77.45,-22.51,849.0,NaN,2021,9,27,0,0,39,3,0
1,Adakamaranahalli,2021-09-27 06:00:00,13.07,77.45,-22.87,849.0,0 days 06:00:00,2021,9,27,6,0,39,3,0
2,Adakamaranahalli,2021-09-27 12:00:00,13.07,77.45,-23.23,849.0,0 days 06:00:00,2021,9,27,12,0,39,3,0
3,Adakamaranahalli,2021-09-27 18:00:00,13.07,77.45,-22.66,849.0,0 days 06:00:00,2021,9,27,18,0,39,3,0
4,Adakamaranahalli,2021-09-28 00:00:00,13.07,77.45,-22.46,849.0,0 days 06:00:00,2021,9,28,0,1,39,3,0


## 4.2 Lag Feature Engineering

Lag features provide the model with historical groundwater information by including previous observations as predictors. Since groundwater levels are sequential and measured every 6 hours, previous readings are expected to have a strong influence on future levels.

In this project, lag features representing the previous 6 hours, 24 hours, and 7 days are created separately for each monitoring station.

In [5]:

# Ensure data is sorted correctly
df = df.sort_values(['Station', 'Data Acquisition Time'])

target = 'Groundwater Level Telemetry 6 Hourly (meter)'

# Previous observation (6 hours)
df['Lag_1'] = df.groupby('Station')[target].shift(1)

# Previous day (4 × 6-hour intervals)
df['Lag_4'] = df.groupby('Station')[target].shift(4)

# Previous week (28 × 6-hour intervals)
df['Lag_28'] = df.groupby('Station')[target].shift(28)

print("Lag features created successfully.")

Lag features created successfully.


In [6]:
df[['Station',
    'Data Acquisition Time',
    target,
    'Lag_1',
    'Lag_4',
    'Lag_28']].head(35)

,Station,Data Acquisition Time,Groundwater Level Telemetry 6 Hourly (meter),Lag_1,Lag_4,Lag_28
0,Adakamaranahalli,2021-09-27 00:00:00,-22.51,NaN,NaN,NaN
1,Adakamaranahalli,2021-09-27 06:00:00,-22.87,-22.51,NaN,NaN
2,Adakamaranahalli,2021-09-27 12:00:00,-23.23,-22.87,NaN,NaN
3,Adakamaranahalli,2021-09-27 18:00:00,-22.66,-23.23,NaN,NaN
4,Adakamaranahalli,2021-09-28 00:00:00,-22.46,-22.66,-22.51,NaN
5,Adakamaranahalli,2021-09-28 06:00:00,-23.42,-22.46,-22.87,NaN
6,Adakamaranahalli,2021-09-28 12:00:00,-22.74,-23.42,-23.23,NaN
7,Adakamaranahalli,2021-09-28 18:00:00,-22.53,-22.74,-22.66,NaN
8,Adakamaranahalli,2021-09-29 00:00:00,-22.38,-22.53,-22.46,NaN
9,Adakamaranahalli,2021-09-29 06:00:00,-22.91,-22.38,-23.42,NaN


### Interpretation

The lag features were successfully created for each monitoring station.

- **Lag_1** represents the groundwater level recorded 6 hours earlier.
- **Lag_4** represents the groundwater level recorded 24 hours earlier.
- **Lag_28** represents the groundwater level recorded approximately one week earlier.

The initial rows for each station contain missing values because sufficient historical observations are unavailable. This behavior is expected and these rows will be removed before model training.

These lag variables provide historical context to the forecasting models and are expected to be among the most informative predictors for groundwater level prediction.

## 4.3 Rolling Window Features

Rolling statistical features summarize recent groundwater behavior over fixed windows. Unlike lag features that use a single previous observation, rolling statistics capture short-term trends and local variability.

The following rolling features are generated independently for each monitoring station:

- Rolling Mean (previous 4 observations)
- Rolling Standard Deviation (previous 4 observations)

These features help machine learning models understand whether groundwater levels are stable, increasing, or highly variable in recent observations.

In [7]:
target = "Groundwater Level Telemetry 6 Hourly (meter)"

grouped = df.groupby("Station")[target]

# Previous 4 observations (24 hours)
df["RollingMean_4"] = (
    grouped.shift(1)
           .rolling(window=4)
           .mean()
)

df["RollingStd_4"] = (
    grouped.shift(1)
           .rolling(window=4)
           .std()
)

print("Rolling window features created successfully.")

Rolling window features created successfully.


In [8]:
df[[
    "Station",
    "Data Acquisition Time",
    target,
    "RollingMean_4",
    "RollingStd_4"
]].head(20)

,Station,Data Acquisition Time,Groundwater Level Telemetry 6 Hourly (meter),RollingMean_4,RollingStd_4
0,Adakamaranahalli,2021-09-27 00:00:00,-22.51,NaN,NaN
1,Adakamaranahalli,2021-09-27 06:00:00,-22.87,NaN,NaN
2,Adakamaranahalli,2021-09-27 12:00:00,-23.23,NaN,NaN
3,Adakamaranahalli,2021-09-27 18:00:00,-22.66,NaN,NaN
4,Adakamaranahalli,2021-09-28 00:00:00,-22.46,-22.8175,0.312130
5,Adakamaranahalli,2021-09-28 06:00:00,-23.42,-22.8050,0.329090
6,Adakamaranahalli,2021-09-28 12:00:00,-22.74,-22.9425,0.455805
7,Adakamaranahalli,2021-09-28 18:00:00,-22.53,-22.8200,0.416973
8,Adakamaranahalli,2021-09-29 00:00:00,-22.38,-22.7875,0.438130
9,Adakamaranahalli,2021-09-29 06:00:00,-22.91,-22.7675,0.459375


### Interpretation

Rolling statistical features were successfully generated for each monitoring station.

- **RollingMean_4** represents the average groundwater level during the previous 24 hours (4 observations).
- **RollingStd_4** measures the variability of groundwater levels over the same period.

These features summarize recent groundwater behavior and help machine learning models distinguish between stable periods and rapidly changing groundwater conditions.

The first four observations of each station contain missing values because there is insufficient historical data to compute a rolling window. These rows will be removed before model training.

## 4.4 Cyclical Time Features

Temporal variables such as **Hour** and **Month** are cyclical rather than linear. For example, December is followed by January, and 18:00 is followed by 00:00.

Instead of using raw numerical values, cyclical encoding transforms these variables using sine and cosine functions. This preserves the continuity of cyclic patterns and enables machine learning models to better capture periodic behavior in groundwater levels.

The following cyclical features are created:

- Hour_sin
- Hour_cos
- Month_sin
- Month_cos

In [9]:
import numpy as np

# Hour (0–23)
df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

# Month (1–12)
df["Month_sin"] = np.sin(2 * np.pi * (df["Month"] - 1) / 12)
df["Month_cos"] = np.cos(2 * np.pi * (df["Month"] - 1) / 12)

print("Cyclical time features created successfully.")

Cyclical time features created successfully.


In [10]:
df[[
    "Data Acquisition Time",
    "Hour",
    "Hour_sin",
    "Hour_cos",
    "Month",
    "Month_sin",
    "Month_cos"
]].head(12)

,Data Acquisition Time,Hour,Hour_sin,Hour_cos,Month,Month_sin,Month_cos
0,2021-09-27 00:00:00,0,0.000000e+00,1.000000e+00,9,-0.866025,-0.5
1,2021-09-27 06:00:00,6,1.000000e+00,6.123234e-17,9,-0.866025,-0.5
2,2021-09-27 12:00:00,12,1.224647e-16,-1.000000e+00,9,-0.866025,-0.5
3,2021-09-27 18:00:00,18,-1.000000e+00,-1.836970e-16,9,-0.866025,-0.5
4,2021-09-28 00:00:00,0,0.000000e+00,1.000000e+00,9,-0.866025,-0.5
5,2021-09-28 06:00:00,6,1.000000e+00,6.123234e-17,9,-0.866025,-0.5
6,2021-09-28 12:00:00,12,1.224647e-16,-1.000000e+00,9,-0.866025,-0.5
7,2021-09-28 18:00:00,18,-1.000000e+00,-1.836970e-16,9,-0.866025,-0.5
8,2021-09-29 00:00:00,0,0.000000e+00,1.000000e+00,9,-0.866025,-0.5
9,2021-09-29 06:00:00,6,1.000000e+00,6.123234e-17,9,-0.866025,-0.5


### Interpretation

The cyclical time features were successfully generated using sine and cosine transformations.

Unlike raw numerical encoding, cyclical encoding preserves the circular nature of temporal variables. For example, Hour 23 and Hour 0 become close to each other in the transformed feature space instead of appearing numerically far apart.

The newly created features are:

- **Hour_sin**
- **Hour_cos**
- **Month_sin**
- **Month_cos**

These variables allow machine learning models to learn periodic daily and seasonal groundwater patterns more effectively.

## 4.5 Station Encoding

Machine learning algorithms require numerical inputs. Therefore, the categorical **Station** variable is converted into numerical labels using Label Encoding.

Each monitoring station is assigned a unique integer identifier while preserving all observations belonging to that station.

This encoded feature will be used during model training, whereas the original station names will be retained for interpretation and visualization.

In [11]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["Station_ID"] = encoder.fit_transform(df["Station"])

print("Station encoding completed successfully.")
print(f"Number of encoded stations: {df['Station_ID'].nunique()}")

Station encoding completed successfully.
Number of encoded stations: 25


In [12]:
# Display station mapping

station_mapping = (
    df[["Station", "Station_ID"]]
    .drop_duplicates()
    .sort_values("Station_ID")
    .reset_index(drop=True)
)

station_mapping

,Station,Station_ID
0,Adakamaranahalli,0
1,Anekal_1,1
2,Attibele_1,2
3,Avalahalli,3
4,Bagalagunte,4
5,Beguru_1,5
6,Byadarahalli,6
7,Chandapura_1,7
8,Devarabeesanahalli_1,8
9,Doddakannahalli,9


### Interpretation

The categorical **Station** feature was successfully converted into numerical labels using Label Encoding.

Each monitoring station has been assigned a unique integer identifier while preserving the original station names for future interpretation.

The encoded station identifier will be used as an input feature during model training, allowing machine learning algorithms to distinguish between different monitoring locations.

## 4.6 Handling Missing Values from Feature Engineering

Lag and rolling statistical features require historical observations. Consequently, the first few records of each monitoring station contain missing values because sufficient prior data is unavailable.

Rather than imputing these engineered missing values, the affected rows are removed. This ensures that every observation used for model training contains complete historical information without introducing artificial estimates.

In [13]:
print("Dataset shape before removing NaNs:", df.shape)

df_model = df.dropna().copy()

print("Dataset shape after removing NaNs:", df_model.shape)

print("\nRows removed:", len(df) - len(df_model))

print("\nRemaining Missing Values:")
print(df_model.isnull().sum().sum())

Dataset shape before removing NaNs: (99445, 25)
Dataset shape after removing NaNs: (69404, 25)

Rows removed: 30041

Remaining Missing Values:
0


In [14]:
print("Final dataset shape:", df_model.shape)

print("\nRemaining missing values:")
print(df_model.isnull().sum())

df_model.head()

Final dataset shape: (69404, 25)

Remaining missing values:
Station                                         0
Data Acquisition Time                           0
Latitude                                        0
Longitude                                       0
Groundwater Level Telemetry 6 Hourly (meter)    0
RL_MSL                                          0
time_diff                                       0
Year                                            0
Month                                           0
Day                                             0
Hour                                            0
DayOfWeek                                       0
WeekOfYear                                      0
Quarter                                         0
IsWeekend                                       0
Lag_1                                           0
Lag_4                                           0
Lag_28                                          0
RollingMean_4                           

,Station,Data Acquisition Time,Latitude,Longitude,Groundwater Level Telemetry 6 Hourly (meter),RL_MSL,time_diff,Year,Month,Day,...,Lag_1,Lag_4,Lag_28,RollingMean_4,RollingStd_4,Hour_sin,Hour_cos,Month_sin,Month_cos,Station_ID
28,Adakamaranahalli,2021-10-06 00:00:00,13.07,77.45,-22.50,849.0,1 days 06:00:00,2021,10,6,...,-22.71,-22.34,-22.51,-22.8625,0.419235,0.000000e+00,1.000000e+00,-1.0,-1.836970e-16,0
29,Adakamaranahalli,2021-10-06 06:00:00,13.07,77.45,-22.83,849.0,0 days 06:00:00,2021,10,6,...,-22.50,-23.16,-22.87,-22.9025,0.355563,1.000000e+00,6.123234e-17,-1.0,-1.836970e-16,0
30,Adakamaranahalli,2021-10-06 12:00:00,13.07,77.45,-22.88,849.0,0 days 06:00:00,2021,10,6,...,-22.83,-23.24,-23.23,-22.8200,0.311448,1.224647e-16,-1.000000e+00,-1.0,-1.836970e-16,0
31,Adakamaranahalli,2021-10-06 18:00:00,13.07,77.45,-22.52,849.0,0 days 06:00:00,2021,10,6,...,-22.88,-22.71,-22.66,-22.7300,0.169115,-1.000000e+00,-1.836970e-16,-1.0,-1.836970e-16,0
32,Adakamaranahalli,2021-10-07 00:00:00,13.07,77.45,-22.35,849.0,0 days 06:00:00,2021,10,7,...,-22.52,-22.50,-22.46,-22.6825,0.200395,0.000000e+00,1.000000e+00,-1.0,-1.836970e-16,0


## Conclusion

Feature engineering was successfully completed for the groundwater level dataset.

The following features were created:

- Temporal features (Year, Month, Day, Hour, DayOfWeek, WeekOfYear, Quarter, IsWeekend)
- Lag features (Lag_1, Lag_4, Lag_28)
- Rolling statistics (RollingMean_4, RollingStd_4)
- Cyclical time features (Hour_sin, Hour_cos, Month_sin, Month_cos)
- Encoded station identifiers (Station_ID)

Rows containing missing values generated during lag and rolling feature creation were removed to ensure a complete dataset for machine learning.

The resulting dataset contains engineered temporal and historical features that capture groundwater dynamics while avoiding data leakage.

This processed dataset will be used in the next notebook for model development and forecasting.

In [15]:
# Save Feature Engineered Dataset
df_model.to_csv("groundwater_feature_engineered.csv", index=False)

print("Feature engineered dataset saved successfully.")
print("Final Shape:", df_model.shape)

Feature engineered dataset saved successfully.
Final Shape: (69404, 25)
